# TT-10 — MLP Classifier: Đọc số viết tay trên séc / phiếu chuyển khoản

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

RANDOM_STATE = 42
%matplotlib inline

## 1. Load dữ liệu — MNIST thật

In [ ]:
X_all, y_all = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y_all = y_all.astype(int)
IMG_SIZE = 28
PIXEL_MAX = 255.0

X_raw, y = X_all, y_all
X_norm = X_raw / PIXEL_MAX

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train_norm, X_test_norm, _, _ = train_test_split(
    X_norm, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Tổng: {X_raw.shape[0]} mẫu, {X_raw.shape[1]} pixel/ảnh ({IMG_SIZE}x{IMG_SIZE}), {len(np.unique(y))} lớp")
print(f"Train: {len(y_train)} | Test: {len(y_test)}")

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_raw[i].reshape(IMG_SIZE, IMG_SIZE), cmap="gray")
    ax.set_title(f"nhãn: {y[i]}")
    ax.axis("off")
plt.suptitle(f"Ví dụ ảnh chữ số {IMG_SIZE}x{IMG_SIZE}")
plt.tight_layout()
plt.show()

## 3. Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
lr.fit(X_train_norm, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test_norm))
print(f"Baseline Logistic Regression: acc = {acc_lr:.4f}")

## 4-5. MLP: không chuẩn hoá vs có chuẩn hoá

In [ ]:
def make_mlp(**kw):
    base = dict(hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
                alpha=1e-4, batch_size=128, learning_rate_init=1e-3, max_iter=300,
                early_stopping=True, n_iter_no_change=10, random_state=RANDOM_STATE)
    base.update(kw)
    return MLPClassifier(**base)

t0 = time.time()
mlp_raw = make_mlp()
mlp_raw.fit(X_train_raw, y_train)
acc_raw = accuracy_score(y_test, mlp_raw.predict(X_test_raw))
print(f"MLP KHÔNG chuẩn hoá: acc = {acc_raw:.4f} (iters={mlp_raw.n_iter_}, {time.time()-t0:.1f}s)")

t0 = time.time()
mlp_norm = make_mlp()
mlp_norm.fit(X_train_norm, y_train)
acc_norm = accuracy_score(y_test, mlp_norm.predict(X_test_norm))
print(f"MLP CÓ chuẩn hoá:    acc = {acc_norm:.4f} (iters={mlp_norm.n_iter_}, {time.time()-t0:.1f}s)")

## 6. So sánh 4 kiến trúc mạng

In [ ]:
architectures = [(64,), (128,), (128, 64), (256, 128, 64)]

def count_params(sizes, n_in=IMG_SIZE*IMG_SIZE, n_out=10):
    layers = [n_in] + list(sizes) + [n_out]
    return sum(layers[i]*layers[i+1] + layers[i+1] for i in range(len(layers)-1))

arch_results = []
for arch in architectures:
    t0 = time.time()
    m = make_mlp(hidden_layer_sizes=arch)
    m.fit(X_train_norm, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_norm))
    dt = time.time() - t0
    n_params = count_params(arch)
    arch_results.append((arch, acc, n_params, dt))
    print(f"{arch}: acc={acc:.4f}, params={n_params}, time={dt:.1f}s")

best_acc, best_arch = max((a, arc) for arc, a, *_ in arch_results), None
best_acc, best_arch = -1, None
for arch, acc, n_params, dt in arch_results:
    if acc > best_acc:
        best_acc, best_arch = acc, arch
print(f"\nKiến trúc tốt nhất: {best_arch} (acc={best_acc:.4f})")

labels = [str(a) for a, *_ in arch_results]
accs = [a for _, a, *_ in arch_results]
params = [p for *_, p, _ in arch_results]

fig, ax1 = plt.subplots(figsize=(7,4.5))
x = np.arange(len(labels))
ax1.bar(x, accs, color="#4C72B0", alpha=0.85)
ax1.set_ylabel("Accuracy"); ax1.set_ylim(0.85, 1.0)
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_xlabel("Kiến trúc")
ax2 = ax1.twinx()
ax2.plot(x, params, color="#C44E52", marker="o")
ax2.set_ylabel("Số tham số")
plt.title("Accuracy vs Số tham số theo kiến trúc")
plt.tight_layout()
plt.show()

## 7. Đường loss theo epoch (mạng chính 128,64)

In [ ]:
main_model = mlp_norm
plt.figure(figsize=(6,4))
plt.plot(main_model.loss_curve_)
plt.title("Loss curve — MLP (128,64), chuẩn hoá")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.grid(alpha=0.3)
plt.show()

## 8. So sánh 3 activation function

In [ ]:
plt.figure(figsize=(6,4))
act_results = {}
for act in ["relu", "tanh", "logistic"]:
    m = make_mlp(activation=act, max_iter=300)
    m.fit(X_train_norm, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_norm))
    act_results[act] = acc
    plt.plot(m.loss_curve_, label=f"{act} (acc={acc:.3f})")
    print(f"activation={act}: acc={acc:.4f}, n_iter={m.n_iter_}")
plt.title("So sánh activation function")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 9. So sánh 3 learning rate

In [ ]:
plt.figure(figsize=(6,4))
for lrate in [1e-2, 1e-3, 1e-4]:
    m = make_mlp(learning_rate_init=lrate, max_iter=300)
    m.fit(X_train_norm, y_train)
    acc = accuracy_score(y_test, m.predict(X_test_norm))
    plt.plot(m.loss_curve_, label=f"lr={lrate} (acc={acc:.3f})")
    print(f"learning_rate={lrate}: acc={acc:.4f}")
plt.title("So sánh learning rate")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 10. Ma trận nhầm lẫn 10x10

In [ ]:
y_pred = main_model.predict(X_test_norm)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel("Dự đoán"); plt.ylabel("Thực tế")
plt.title("Ma trận nhầm lẫn 10x10")
for i in range(10):
    for j in range(10):
        if cm[i,j] > 0:
            plt.text(j, i, cm[i,j], ha="center", va="center",
                      color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=8)
plt.tight_layout()
plt.show()

cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
idx = np.unravel_index(np.argmax(cm_off), cm_off.shape)
print(f"Cặp số hay nhầm nhất: thực tế={idx[0]} -> dự đoán={idx[1]} ({cm_off[idx]} lần)")

## 11. Các ảnh dự đoán sai

In [ ]:
wrong_idx = np.where(y_pred != y_test)[0][:20]
n_show = len(wrong_idx)
if n_show > 0:
    cols = 5
    rows = (n_show + cols - 1) // cols
    plt.figure(figsize=(cols*2, rows*2.2))
    for i, widx in enumerate(wrong_idx):
        plt.subplot(rows, cols, i+1)
        plt.imshow(X_test_raw[widx].reshape(IMG_SIZE, IMG_SIZE), cmap="gray")
        plt.title(f"thật:{y_test[widx]} đoán:{y_pred[widx]}", fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()
print(f"Số ảnh sai (tối đa 20 hiển thị): {n_show}")

## 12. Cơ chế human-in-the-loop — ngưỡng tin cậy 99%

In [ ]:
proba = main_model.predict_proba(X_test_norm)
confidence = proba.max(axis=1)
pred = proba.argmax(axis=1)

THRESH = 0.99
auto_mask = confidence >= THRESH
n_auto = auto_mask.sum()
n_manual = (~auto_mask).sum()
acc_auto = accuracy_score(y_test[auto_mask], pred[auto_mask]) if n_auto > 0 else float("nan")
acc_overall = accuracy_score(y_test, pred)

print(f"Tổng test: {len(y_test)}")
print(f"Tự động xử lý (conf>=99%): {n_auto} ({100*n_auto/len(y_test):.1f}%) — accuracy: {acc_auto:.4f}")
print(f"Chuyển người kiểm tra:     {n_manual} ({100*n_manual/len(y_test):.1f}%)")
print(f"Accuracy tổng thể (không lọc): {acc_overall:.4f}")

## Lưu model pipeline để triển khai

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", make_mlp(hidden_layer_sizes=best_arch)),
])
pipeline.fit(X_train_raw, y_train)
acc_pipeline = accuracy_score(y_test, pipeline.predict(X_test_raw))
joblib.dump(pipeline, "../models/mlp_pipeline.joblib")
print(f"Pipeline (StandardScaler + MLP {best_arch}) acc={acc_pipeline:.4f} — đã lưu models/mlp_pipeline.joblib")